In [0]:
import time
import numpy as np
import pandas as pd
import mlflow
from datetime import datetime, timezone

t0 = time.time()
run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# ---------- Data inputs from our tables ----------
px = spark.sql("""SELECT price_date, price FROM oil_shock.gold.fct_price_daily
                  WHERE series_id = 'RBRTE' ORDER BY price_date""").toPandas()
px["price_date"] = pd.to_datetime(px["price_date"])
p_now = px["price"].tail(10).median()          # 10-day median: robust to one-day spikes (e.g. Oct 2)
p_now_date = px["price_date"].max().date()
p_prewar = px.loc[(px["price_date"] >= "2026-02-01") & (px["price_date"] <= "2026-02-27"), "price"].mean()

hz = spark.sql("""SELECT transit_date, ships_tanker_pct_of_2025 AS pct FROM oil_shock.gold.fct_chokepoint_daily
                  WHERE lower(chokepoint_name) LIKE '%hormuz%' AND ships_tanker_pct_of_2025 IS NOT NULL
                  ORDER BY transit_date""").toPandas()
h_now = float(hz["pct"].iloc[-1]) / 100
h_date = hz["transit_date"].iloc[-1]

# ---------- Assumptions (all sourced) ----------
F = 19.87            # Hormuz flow 2025, mb/d (IEA/Kpler: crude 14.95 + products 4.93)
BYPASS = 4.5         # spare bypass in use, mb/d (IEA range 3.5-5.5; mid)
BYPASS_EW = 4.0      # of which Saudi East-West (IEA: 3-5 spare)
STOCK_NOW = 100 / 120          # G7 100m barrels over ~4 months -> ~0.83 mb/d
D = 104.40           # world demand 2025, mb/d (EIA STEO Oct 2026)
GOM = 1.81           # US Gulf of Mexico output, mb/d (EIA)
HURRICANE_SHARE = 0.5          # assumed share shut in by a major hurricane (adjustable)
ELASTICITY = {"high_sensitivity": 0.16,   # Caldara et al. 2019, IV targets (0.081 + 0.080)
              "low_sensitivity": 0.29}    # Kilian 2019 (supply <= 0.04, demand -0.25)
LIT_BASELINE = 0.24                       # Caldara et al. 2019 baseline (0.10 + 0.14)

def net_loss(h, stocks_on=True, ew_ok=True, extra=0.0):
    flow_lost = F * (1 - np.asarray(h, dtype=float))
    bypass_cap = BYPASS if ew_ok else BYPASS - BYPASS_EW
    bypass = np.minimum(bypass_cap, flow_lost)
    stocks = STOCK_NOW if stocks_on else 0.0
    return np.maximum(flow_lost - bypass - stocks, 0.0) + extra

loss_now = float(net_loss(h_now))
eps_mid = (loss_now / D) / np.log(p_now / p_prewar)   # elasticity implied by the war itself
ELASTICITY["middle_war_implied"] = eps_mid
in_range = 0.16 <= eps_mid <= 0.29

def price(loss, eps):
    return p_now * np.exp((np.asarray(loss) - loss_now) / (D * eps))

# ---------- Grid for Power BI (reopening 0-100% x stocks x pipeline x sensitivity) ----------
H = np.round(np.arange(0, 1.0001, 0.05), 2)
grid = []
for stocks_on in (True, False):
    for ew_ok in (True, False):
        loss = net_loss(H, stocks_on, ew_ok)
        for case, eps in ELASTICITY.items():
            for h, l, p in zip(H, loss, price(loss, eps)):
                grid.append({"hormuz_open_pct": int(round(h * 100)), "stock_releases_on": stocks_on,
                             "east_west_pipeline_ok": ew_ok, "sensitivity_case": case,
                             "elasticity": round(eps, 3), "net_loss_mbd": round(float(l), 2),
                             "brent_usd": round(float(p), 1)})
grid = pd.DataFrame(grid)

# ---------- Named scenarios ----------
named_specs = [
    ("Today (calibration point)", h_now, True, True, 0.0),
    ("Hormuz reopens to 25%", 0.25, True, True, 0.0),
    ("Hormuz reopens to 50%", 0.50, True, True, 0.0),
    ("Hormuz fully reopens", 1.00, True, True, 0.0),
    ("Emergency stock releases end", h_now, False, True, 0.0),
    ("Saudi East-West pipeline out again", h_now, True, False, 0.0),
    ("Major US Gulf hurricane", h_now, True, True, GOM * HURRICANE_SHARE),
]
named = []
for name, h, s, e, x in named_specs:
    l = float(net_loss(h, s, e, x))
    row = {"scenario": name, "hormuz_open_pct": round(h * 100, 1), "net_loss_mbd": round(l, 2),
           "loss_pct_of_world": round(100 * l / D, 1)}
    for case in ["high_sensitivity", "middle_war_implied", "low_sensitivity"]:
        row[f"brent_{case}"] = round(float(price(l, ELASTICITY[case])), 1)
    named.append(row)
named = pd.DataFrame(named)

assumptions = pd.DataFrame([
    ("Hormuz flow 2025 (mb/d)", F, "IEA Strait of Hormuz page (Kpler): crude 14.95 + products 4.93"),
    ("Spare bypass capacity in use (mb/d)", BYPASS, "IEA: 3.5-5.5 mb/d available; mid used"),
    ("of which Saudi East-West (mb/d)", BYPASS_EW, "IEA: 3-5 mb/d spare"),
    ("Emergency stock release now (mb/d)", round(STOCK_NOW, 2), "G7 Oct 2 2026: 100m barrels over 4 months"),
    ("World demand 2025 (mb/d)", D, "EIA STEO October 2026, table 3e"),
    ("US Gulf of Mexico output (mb/d)", GOM, "EIA Today in Energy (2026 forecast)"),
    ("Hurricane shut-in share", HURRICANE_SHARE, "Assumption, adjustable"),
    ("Elasticity: high sensitivity", 0.16, "Caldara, Cavallo, Iacoviello (2019) JME, IV targets 0.081 + 0.080"),
    ("Elasticity: literature baseline", LIT_BASELINE, "Caldara, Cavallo, Iacoviello (2019) JME, baseline 0.10 + 0.14"),
    ("Elasticity: low sensitivity", 0.29, "Kilian (2019), Dallas Fed WP 1907: supply <= 0.04, demand -0.25"),
    ("Elasticity: war-implied (middle)", round(eps_mid, 3), "Calibrated: today's net loss vs Brent change since Feb 2026"),
    ("Brent now (USD, 10-day median)", round(p_now, 2), f"EIA RBRTE, latest {p_now_date}"),
    ("Brent pre-war (USD, Feb 1-27 2026 avg)", round(p_prewar, 2), "EIA RBRTE"),
    ("Hormuz open now (% of 2025 tanker transits)", round(h_now * 100, 1), f"IMF PortWatch, {h_date}"),
], columns=["assumption", "value", "source"])

for nm, out in {"scenario_assumptions": assumptions, "scenario_grid": grid, "scenario_named": named}.items():
    out = out.copy(); out["run_at_utc"] = run_stamp
    spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"oil_shock.ml.{nm}")

mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-scenarios")
with mlflow.start_run(run_name=f"scenario_calculator {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/44_scenarios", "project": "oil-shock-2026", "author": "Zia Malik",
                     "method": "net supply loss -> price via short-run elasticity; transparent calculator"})
    mlflow.log_params({r["assumption"]: r["value"] for _, r in assumptions.iterrows()})
    mlflow.log_metrics({"eps_war_implied": float(eps_mid), "eps_in_literature_range": float(in_range),
                        "net_loss_now_mbd": loss_now})

# ---------- Compact summary to paste ----------
print(f"Done in {time.time() - t0:.0f}s | grid rows: {len(grid)}")
print(f"Brent now ${p_now:.1f} ({p_now_date}) | pre-war ${p_prewar:.1f} | Hormuz open {h_now*100:.1f}% ({h_date})")
print(f"Net loss now {loss_now:.2f} mb/d = {100*loss_now/D:.1f}% of world demand")
print(f"War-implied elasticity {eps_mid:.3f} | literature 0.16-0.29 -> {'INSIDE' if in_range else 'OUTSIDE'}")
print(named.to_string(index=False))